In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB

# ===============================
# 1. Load Data
# ===============================
df = pd.read_csv("/content/selected_telco_churn_columns.csv")

# Keep only needed columns
df = df[['InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']]

# ===============================
# 2. Split Features & Target
# ===============================
X = df[['InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod']]
y = df['Churn']  # Keep as Yes/No (strings) for training

# ===============================
# 3. Encode Features
# ===============================
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X)

# ===============================
# 4. Train/Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# ===============================
# 5. Train Naive Bayes Model
# ===============================
model = CategoricalNB()
model.fit(X_train, y_train)

# ===============================
# 6. Evaluate Accuracy
# ===============================
accuracy = model.score(X_test, y_test)
print(f"Model Accuracy on Test Set: {accuracy:.4f}")

# ===============================
# 7. Predict on New Data
# ===============================
new_data = pd.DataFrame([{
    "InternetService": "Fiber optic",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check"
}])

# Encode using the same encoder
new_data_encoded = encoder.transform(new_data)

# Predict
new_prediction = model.predict(new_data_encoded)[0]
new_proba = model.predict_proba(new_data_encoded)[0]

print("\n=== New Data Prediction ===")
print(new_data)
print(f"Predicted Churn: {new_prediction}")
print(f"Probability Yes: {new_proba[list(model.classes_).index('Yes')]:.4f}")
print(f"Probability No:  {new_proba[list(model.classes_).index('No')]:.4f}")


Model Accuracy on Test Set: 0.7757

=== New Data Prediction ===
  InternetService        Contract PaperlessBilling     PaymentMethod
0     Fiber optic  Month-to-month              Yes  Electronic check
Predicted Churn: Yes
Probability Yes: 0.8218
Probability No:  0.1782


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv('/content/selected_telco_churn_columns.csv')

# ===============================
# TRAINING PART (scikit-learn)
# ===============================

# Separate features and target (keep target as strings for now)
X = df.drop(columns=['Churn'])
y = df['Churn']  # still "Yes" / "No"

# Encode only the features, not the target
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X)

# Split data into train and test (model has no idea what "Yes" or "No" means yet)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Train Naive Bayes
model = CategoricalNB()
model.fit(X_train, y_train)  # y is still "Yes"/"No" strings

# Predict and evaluate
y_pred = model.predict(X_test)
print("=== Naive Bayes Model Accuracy ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ===============================
# MANUAL STEP-BY-STEP CALCULATION
# ===============================

# Select example row to classify
example_index = 0
example_row = df.iloc[example_index, :-1]  # independent variables only
print("\nExample row to classify:\n", example_row, "\n")

# Step 1: Frequency and Likelihood tables for all features
frequency_tables = {}
likelihood_tables = {}

total_yes = sum(df['Churn'] == 'Yes')
total_no = sum(df['Churn'] == 'No')
total_records = len(df)

for col in df.columns[:-1]:
    freq_table = pd.crosstab(df[col], df['Churn'])
    frequency_tables[col] = freq_table
    like_table = freq_table.copy().astype(float)
    like_table['Yes'] = like_table['Yes'] / total_yes
    like_table['No'] = like_table['No'] / total_no
    likelihood_tables[col] = like_table

# Step 2: Priors
p_yes = total_yes / total_records
p_no = total_no / total_records

# Step 3: Naive Bayes calculation for the example row
calculation_steps = []
p_x_given_yes = 1
p_x_given_no = 1

for col, value in example_row.items():
    freq = frequency_tables[col]
    like = likelihood_tables[col]

    count_yes = freq.loc[value, 'Yes']
    count_no = freq.loc[value, 'No']
    p_x = (count_yes + count_no) / total_records

    prob_yes = like.loc[value, 'Yes']
    prob_no = like.loc[value, 'No']

    p_x_given_yes *= prob_yes
    p_x_given_no *= prob_no

    calculation_steps.append({
        'Feature': col,
        'Value': value,
        'Frequency Table': freq,
        'Likelihood Table': like,
        'P(Value|Yes)': prob_yes,
        'P(Value|No)': prob_no,
        'P(Value)': p_x
    })

# Step 4: Multiply by priors
posterior_yes = p_x_given_yes * p_yes
posterior_no = p_x_given_no * p_no

# Step 5: Normalize to get posterior probabilities
evidence = posterior_yes + posterior_no
posterior_yes_norm = posterior_yes / evidence
posterior_no_norm = posterior_no / evidence

# Step 6: Display results
for step in calculation_steps:
    print(f"=== {step['Feature']} ===")
    print("\nFrequency Table:")
    print(step['Frequency Table'])
    print("\nLikelihood Table:")
    print(step['Likelihood Table'])
    print(f"\nP({step['Value']}|Yes) = {step['P(Value|Yes)']:.4f}")
    print(f"P({step['Value']}|No) = {step['P(Value|No)']:.4f}")
    print(f"P({step['Value']}) = {step['P(Value)']:.4f}")
    print("-"*50)

# Step 7: Final results
print("\n=== Final Naive Bayes Manual Calculation ===")
print(f"P(x|Yes) = {p_x_given_yes:.6f}")
print(f"P(x|No) = {p_x_given_no:.6f}")
print(f"P(Yes) = {p_yes:.4f}, P(No) = {p_no:.4f}")
print(f"Posterior P(Yes|x) = {posterior_yes_norm:.4f}")
print(f"Posterior P(No|x) = {posterior_no_norm:.4f}")

# Model prediction for same example (model still predicts "Yes" or "No")
example_encoded = encoder.transform([example_row])
model_prediction = model.predict(example_encoded)[0]
print(f"\nModel Prediction for the example row: {model_prediction}")


=== Naive Bayes Model Accuracy ===
Accuracy: 0.7757274662881476

Classification Report:
              precision    recall  f1-score   support

          No       0.88      0.81      0.84      1036
         Yes       0.56      0.69      0.62       373

    accuracy                           0.78      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.80      0.78      0.78      1409


Example row to classify:
 InternetService                  DSL
Contract              Month-to-month
PaperlessBilling                 Yes
PaymentMethod       Electronic check
Name: 0, dtype: object 

=== InternetService ===

Frequency Table:
Churn              No   Yes
InternetService            
DSL              1962   459
Fiber optic      1799  1297
No               1413   113

Likelihood Table:
Churn                  No       Yes
InternetService                    
DSL              0.379204  0.245586
Fiber optic      0.347700  0.693954
No               0.273096  0.060460

P(D

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report

# ===============================
# Load dataset
# ===============================
df = pd.read_csv('/content/selected_telco_churn_columns.csv')

# ===============================
# TRAINING PART (scikit-learn)
# ===============================
X = df.drop(columns=['Churn'])
y = df['Churn']  # still "Yes" / "No"

encoder = OrdinalEncoder()
# Fit the encoder on the values (NumPy array) to avoid the UserWarning
X_encoded = encoder.fit_transform(X.values)

# Split train/test (model has no idea what Yes/No means yet)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

model = CategoricalNB()
model.fit(X_train, y_train)

# Predict & evaluate
y_pred = model.predict(X_test)
print("=== Naive Bayes Model Accuracy ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ===============================
# MANUAL STEP-BY-STEP CALCULATION
# ===============================
example_index = 0
example_row = df.iloc[example_index, :-1]  # features only
print("\nExample row to classify:\n", example_row, "\n")

frequency_tables = {}
likelihood_tables = {}
total_yes = sum(df['Churn'] == 'Yes')
total_no = sum(df['Churn'] == 'No')
total_records = len(df)

for col in df.columns[:-1]:
    freq_table = pd.crosstab(df[col], df['Churn'])
    frequency_tables[col] = freq_table
    like_table = freq_table.copy().astype(float)
    like_table['Yes'] = like_table['Yes'] / total_yes
    like_table['No'] = like_table['No'] / total_no
    likelihood_tables[col] = like_table

p_yes = total_yes / total_records
p_no = total_no / total_records

p_x_given_yes = 1
p_x_given_no = 1
calculation_steps = []

for col, value in example_row.items():
    freq = frequency_tables[col]
    like = likelihood_tables[col]

    count_yes = freq.loc[value, 'Yes']
    count_no = freq.loc[value, 'No']
    p_x = (count_yes + count_no) / total_records

    prob_yes = like.loc[value, 'Yes']
    prob_no = like.loc[value, 'No']

    p_x_given_yes *= prob_yes
    p_x_given_no *= prob_no

    calculation_steps.append({
        'Feature': col,
        'Value': value,
        'Frequency Table': freq,
        'Likelihood Table': like,
        'P(Value|Yes)': prob_yes,
        'P(Value|No)': prob_no,
        'P(Value)': p_x
    })

posterior_yes = p_x_given_yes * p_yes
posterior_no = p_x_given_no * p_no
evidence = posterior_yes + posterior_no
posterior_yes_norm = posterior_yes / evidence
posterior_no_norm = posterior_no / evidence

# Display step-by-step
for step in calculation_steps:
    print(f"=== {step['Feature']} ===")
    print("\nFrequency Table:")
    print(step['Frequency Table'])
    print("\nLikelihood Table:")
    print(step['Likelihood Table'])
    print(f"\nP({step['Value']}|Yes) = {step['P(Value|Yes)']:.4f}")
    print(f"P({step['Value']}|No) = {step['P(Value|No)']:.4f}")
    print(f"P({step['Value']}) = {step['P(Value)']:.4f}")
    print("-"*50)

print("\n=== Final Naive Bayes Manual Calculation ===")
print(f"P(x|Yes) = {p_x_given_yes:.6f}")
print(f"P(x|No) = {p_x_given_no:.6f}")
print(f"P(Yes) = {p_yes:.4f}, P(No) = {p_no:.4f}")
print(f"Posterior P(Yes|x) = {posterior_yes_norm:.4f}")
print(f"Posterior P(No|x) = {posterior_no_norm:.4f}")

example_encoded = encoder.transform([example_row])
model_prediction = model.predict(example_encoded)[0]
print(f"\nModel Prediction for the example row: {model_prediction}")

# ===============================
# NEW DATA PREDICTION
# ===============================
new_data = pd.DataFrame([{
    "InternetService": "Fiber optic",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check"
}])

new_data_encoded = encoder.transform(new_data.values) # Use .values here
prediction = model.predict(new_data_encoded)


print("\n=== New Data Prediction ===")
print(new_data)
print(f"Predicted Churn: {prediction[0]}")
print(f"Probability Yes: {model.predict_proba(new_data_encoded)[0][list(model.classes_).index('Yes')]:.4f}")
print(f"Probability No:  {model.predict_proba(new_data_encoded)[0][list(model.classes_).index('No')]:.4f}")

=== Naive Bayes Model Accuracy ===
Accuracy: 0.7757274662881476

Classification Report:
              precision    recall  f1-score   support

          No       0.88      0.81      0.84      1036
         Yes       0.56      0.69      0.62       373

    accuracy                           0.78      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.80      0.78      0.78      1409


Example row to classify:
 InternetService                  DSL
Contract              Month-to-month
PaperlessBilling                 Yes
PaymentMethod       Electronic check
Name: 0, dtype: object 

=== InternetService ===

Frequency Table:
Churn              No   Yes
InternetService            
DSL              1962   459
Fiber optic      1799  1297
No               1413   113

Likelihood Table:
Churn                  No       Yes
InternetService                    
DSL              0.379204  0.245586
Fiber optic      0.347700  0.693954
No               0.273096  0.060460

P(D